In [ ]:
from pathlib import Path
import json
import polars as pl

BASE_PATH = Path.cwd()

REPORTS_DIR_PATH = BASE_PATH.joinpath("reports")

In [48]:
judge_models = [
    "grok_4_1_fast_free",
    "openrouter_deepseek_deepseek-chat-v3-1",
    "openrouter_moonshotai_kimi-k2-0905",
]

results = []

for judge_model in judge_models:
    for file in REPORTS_DIR_PATH.joinpath(judge_model).glob("*.json"):
        data = json.loads(file.read_text())
        modelo_respuesta, sistema = data["name"].split("|")
        for case in data["report"]["cases"]:
            _, numero_pregunta, numero_subpregunta = case["case_name"].replace("SP", "").replace("P","").split("-")
            print(f"Procesando: Juez={judge_model}, Sistema={sistema}, Modelo Respuesta={modelo_respuesta}, Pregunta={numero_pregunta}, Subpregunta={numero_subpregunta}")
            respuesta_correcta = next(filter(lambda x: x["name"] == "respuesta_correcta", case["case_scores"]))
            precision_legal = next(filter(lambda x: x["name"] == "precision_legal", case["case_scores"]))
            results.append(
                {
                    "numero_pregunta": int(numero_pregunta),
                    "numero_subpregunta": int(numero_subpregunta),
                    "modelo_juez": judge_model,
                    "sistema": sistema,
                    "modelo_respuesta": modelo_respuesta,
                    "score_respuesta_correcta": respuesta_correcta["value"],
                    "reason_respuesta_correcta": respuesta_correcta["reason"],
                    "score_precision_legal": precision_legal["value"],
                    "reason_precision_legal": precision_legal["reason"],
                }
            )


df = pl.DataFrame(results)
df.write_csv(BASE_PATH.joinpath("final_judge_results.csv"))

Procesando: Juez=grok_4_1_fast_free, Sistema=llm_responses_no_rag, Modelo Respuesta=gpt-5-nano, Pregunta=01, Subpregunta=01
Procesando: Juez=grok_4_1_fast_free, Sistema=llm_responses_no_rag, Modelo Respuesta=gpt-5-nano, Pregunta=01, Subpregunta=02
Procesando: Juez=grok_4_1_fast_free, Sistema=llm_responses_no_rag, Modelo Respuesta=gpt-5-nano, Pregunta=01, Subpregunta=03
Procesando: Juez=grok_4_1_fast_free, Sistema=llm_responses_no_rag, Modelo Respuesta=gpt-5-nano, Pregunta=02, Subpregunta=01
Procesando: Juez=grok_4_1_fast_free, Sistema=llm_responses_no_rag, Modelo Respuesta=gpt-5-nano, Pregunta=02, Subpregunta=02
Procesando: Juez=grok_4_1_fast_free, Sistema=llm_responses_no_rag, Modelo Respuesta=gpt-5-nano, Pregunta=02, Subpregunta=03
Procesando: Juez=grok_4_1_fast_free, Sistema=llm_responses_no_rag, Modelo Respuesta=gpt-5-nano, Pregunta=03, Subpregunta=01
Procesando: Juez=grok_4_1_fast_free, Sistema=llm_responses_no_rag, Modelo Respuesta=gpt-5-nano, Pregunta=03, Subpregunta=02
Procesan